In [3]:
# Імпорт необхідних бібліотек для роботи з HTTP-запитами, архівами та аналізу даних
import requests
import zipfile
import io
import pandas as pd

In [4]:
# Автоматизоване завантаження набору даних Stack Overflow Survey 2025 безпосередньо з офіційного джерела
# Використовуємо io.BytesIO для обробки архіву в пам'яті без збереження на диск
url = "https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-2025.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

In [5]:
# Читання CSV-файлу та створення DataFrame pandas
with z.open("survey_results_schema.csv") as f:schema = pd.read_csv(f)
schema.head()

,qid,qname,question,type,sub,sq_id
0,QID18,TechEndorse_1,What attracts you to a technology or causes yo...,RO,AI integration or AI Agent capabilities,1.0
1,QID18,TechEndorse_2,What attracts you to a technology or causes yo...,RO,Easy-to-use API,2.0
2,QID18,TechEndorse_3,What attracts you to a technology or causes yo...,RO,Robust and complete API,3.0
3,QID18,TechEndorse_4,What attracts you to a technology or causes yo...,RO,Customizable and manageable codebase,4.0
4,QID18,TechEndorse_5,What attracts you to a technology or causes yo...,RO,Reputation for quality,5.0


In [6]:
with z.open("survey_results_public.csv") as f:df = pd.read_csv(f, low_memory=False)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), object(119)
memory usage: 64.6+ MB
None


In [7]:
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(20))

AIAgentObsWrite         48927
SOTagsWant Entry        48761
SOTagsHaveEntry         48733
AIModelsWantEntry       48716
AIAgentOrchWrite        48713
JobSatPoints_15_TEXT    48527
AIAgentKnowWrite        48425
AIModelsHaveEntry       48415
SO_Actions_15_TEXT      48368
AIAgentExtWrite         48332
CommPlatformWantEntr    48007
CommPlatformHaveEntr    47715
DatabaseWantEntry       47658
OfficeStackWantEntry    47574
TechOppose_15_TEXT      47544
TechEndorse_13_TEXT     47182
DevEnvWantEntry         47079
DatabaseHaveEntry       47041
OfficeStackHaveEntry    46602
WebframeWantEntry       46561
dtype: int64


In [8]:
df.columns.tolist()

['ResponseId',
 'MainBranch',
 'Age',
 'EdLevel',
 'Employment',
 'EmploymentAddl',
 'WorkExp',
 'LearnCodeChoose',
 'LearnCode',
 'LearnCodeAI',
 'AILearnHow',
 'YearsCode',
 'DevType',
 'OrgSize',
 'ICorPM',
 'RemoteWork',
 'PurchaseInfluence',
 'TechEndorseIntro',
 'TechEndorse_1',
 'TechEndorse_2',
 'TechEndorse_3',
 'TechEndorse_4',
 'TechEndorse_5',
 'TechEndorse_6',
 'TechEndorse_7',
 'TechEndorse_8',
 'TechEndorse_9',
 'TechEndorse_13',
 'TechEndorse_13_TEXT',
 'TechOppose_1',
 'TechOppose_2',
 'TechOppose_3',
 'TechOppose_5',
 'TechOppose_7',
 'TechOppose_9',
 'TechOppose_11',
 'TechOppose_13',
 'TechOppose_16',
 'TechOppose_15',
 'TechOppose_15_TEXT',
 'Industry',
 'JobSatPoints_1',
 'JobSatPoints_2',
 'JobSatPoints_3',
 'JobSatPoints_4',
 'JobSatPoints_5',
 'JobSatPoints_6',
 'JobSatPoints_7',
 'JobSatPoints_8',
 'JobSatPoints_9',
 'JobSatPoints_10',
 'JobSatPoints_11',
 'JobSatPoints_13',
 'JobSatPoints_14',
 'JobSatPoints_15',
 'JobSatPoints_16',
 'JobSatPoints_15_TEXT',
 

In [9]:
total_respondents = df.shape[0]
total_respondents

49191

In [10]:
qnames = schema['qname'].dropna().tolist()
len(qnames), qnames[:5]

(139,
 ['TechEndorse_1',
  'TechEndorse_2',
  'TechEndorse_3',
  'TechEndorse_4',
  'TechEndorse_5'])

In [11]:
schema_q = set(qnames)
df_cols = set(df.columns)
common_cols = schema_q.intersection(df_cols)
len(common_cols)

126

In [12]:
df_questions = df[list(common_cols)]
df_questions.shape

(49191, 126)

In [13]:
complete_respondents = df_questions.dropna()
complete_respondents.shape

(0, 126)

In [14]:
total_complete = complete_respondents.shape[0]
total_complete

0

In [16]:
'WorkExp' in df.columns

True

In [17]:
df['WorkExp'].head()

0     8.0
1     2.0
2    10.0
3     4.0
4    21.0
Name: WorkExp, dtype: float64

In [18]:
df['WorkExp'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 49191 entries, 0 to 49190
Series name: WorkExp
Non-Null Count  Dtype  
--------------  -----  
42893 non-null  float64
dtypes: float64(1)
memory usage: 384.4 KB


In [19]:
df['WorkExp'].isna().sum().item()

6298

In [20]:
# Розрахунок основних статистичних показників досвіду програмування (mean, median, mode)
mean_exp = df['WorkExp'].mean()
median_exp = df['WorkExp'].median()
mode_exp = df['WorkExp'].mode()[0]
stats = pd.Series({'Mean': mean_exp,'Median': median_exp,'Mode': mode_exp})
stats

Mean      13.367403
Median    10.000000
Mode      10.000000
dtype: float64

In [21]:
'RemoteWork' in df.columns

True

In [22]:
df['RemoteWork'].unique()

array(['Remote', 'Hybrid (some in-person, leans heavy to flexibility)',
       nan, 'In-person', 'Hybrid (some remote, leans heavy to in-person)',
       'Your choice (very flexible, you can come in when you want or just as needed)'],
      dtype=object)

In [23]:
remote_count = df[df['RemoteWork'] == 'Remote'].shape[0]
remote_count

10931

In [24]:
df['RemoteWork'].value_counts()

RemoteWork
Remote                                                                          10931
Hybrid (some remote, leans heavy to in-person)                                   6732
In-person                                                                        6042
Hybrid (some in-person, leans heavy to flexibility)                              5831
Your choice (very flexible, you can come in when you want or just as needed)     4244
Name: count, dtype: int64

In [25]:
df['IsPython'] = df['LanguageHaveWorkedWith'].str.contains('Python', na=False)

In [26]:
'LanguageHaveWorkedWith' in df.columns

True

In [27]:
python_percent = (df['IsPython'].sum() / df.shape[0]) * 100
print(f"Відсоток Python-розробників: {python_percent:.2f}%")

Відсоток Python-розробників: 37.54%


In [28]:
python_count = df['LanguageHaveWorkedWith'].str.contains('Python', na=False).sum()
print("Кількість респондентів, що використовують Python:", python_count)

Кількість респондентів, що використовують Python: 18466


In [29]:
total_respondents = df.shape[0]
total_respondents

49191

In [30]:
python_percent = (python_count / total_respondents) * 100
python_percent = round(python_percent, 1)
print(f"\nВідсоток респондентів, які програмують на Python: {python_percent}%")


Відсоток респондентів, які програмують на Python: 37.5%


In [31]:
df['LearnCode']

0        Online Courses or Certification (includes all ...
1        Online Courses or Certification (includes all ...
2        Online Courses or Certification (includes all ...
3        Other online resources (e.g. standard search, ...
4                                                      NaN
                               ...                        
49186                                                  NaN
49187    Online Courses or Certification (includes all ...
49188    Online Courses or Certification (includes all ...
49189    Videos (not associated with specific online co...
49190    Online Courses or Certification (includes all ...
Name: LearnCode, Length: 49191, dtype: object

In [32]:
df['LearnCode'].dropna().unique()

array(['Online Courses or Certification (includes all media types);Other online resources (e.g. standard search, forum, online community)',
       'Online Courses or Certification (includes all media types);Other online resources (e.g. standard search, forum, online community);Books / Physical media;Videos (not associated with specific online course or certification);Stack Overflow or Stack Exchange',
       'Online Courses or Certification (includes all media types);Videos (not associated with specific online course or certification);Technical documentation (is generated for/by the tool or system)',
       ...,
       'Other online resources (e.g. standard search, forum, online community);Books / Physical media;Videos (not associated with specific online course or certification);Stack Overflow or Stack Exchange;Technical documentation (is generated for/by the tool or system);Colleague or on-the-job training;Blogs or podcasts;Coding Bootcamp;Games or coding challenges;School (i.e., Uni

In [33]:
online_count = df['LearnCode'].str.contains('Online Courses', na=False).sum().item()
online_count

10973

In [34]:
df = df[df['LanguagesHaveEntry'].str.contains('Python', na=False)]
df.shape

(44, 173)

In [35]:
python_salary = df[['Country', 'ConvertedCompYearly']]
python_salary

,Country,ConvertedCompYearly
701,United States of America,273000.0
2731,United States of America,95000.0
2846,United States of America,20000.0
3242,Taiwan,NaN
4550,Ukraine,5300.0
6495,United Kingdom of Great Britain and Northern I...,63986.0
6601,United States of America,145000.0
11434,Netherlands,63808.0
11782,Kenya,NaN
13996,United States of America,165000.0


In [36]:
python_salary = python_salary.dropna(subset=['ConvertedCompYearly'])
python_salary.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 701 to 47946
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              25 non-null     object 
 1   ConvertedCompYearly  25 non-null     float64
dtypes: float64(1), object(1)
memory usage: 600.0+ bytes


In [37]:
salary_by_country = python_salary.groupby('Country')['ConvertedCompYearly'].agg(['mean', 'median'])
salary_by_country

,mean,median
Country,,
Argentina,80000.000000,80000.0
Australia,16252.000000,16252.0
Bangladesh,1636.000000,1636.0
Brazil,1365.000000,1365.0
Germany,127616.000000,127616.0
India,6974.000000,6974.0
Netherlands,63808.000000,63808.0
Portugal,54527.000000,54527.0
Serbia,23759.000000,23759.0


In [38]:
salary_by_country.shape
salary_by_country.head()

,mean,median
Country,,
Argentina,80000.0,80000.0
Australia,16252.0,16252.0
Bangladesh,1636.0,1636.0
Brazil,1365.0,1365.0
Germany,127616.0,127616.0


In [39]:
salary_by_country = salary_by_country.rename(columns={'mean': 'AverageCompensation','median': 'MedianCompensation'})
salary_by_country

,AverageCompensation,MedianCompensation
Country,,
Argentina,80000.000000,80000.0
Australia,16252.000000,16252.0
Bangladesh,1636.000000,1636.0
Brazil,1365.000000,1365.0
Germany,127616.000000,127616.0
India,6974.000000,6974.0
Netherlands,63808.000000,63808.0
Portugal,54527.000000,54527.0
Serbia,23759.000000,23759.0


In [40]:
salary_by_country = salary_by_country.sort_values(by='AverageCompensation',ascending=False)
salary_by_country

,AverageCompensation,MedianCompensation
Country,,
United States of America,164916.666667,159500.0
Germany,127616.000000,127616.0
Argentina,80000.000000,80000.0
Netherlands,63808.000000,63808.0
United Kingdom of Great Britain and Northern Ireland,62171.000000,62625.0
Portugal,54527.000000,54527.0
Serbia,23759.000000,23759.0
Australia,16252.000000,16252.0
India,6974.000000,6974.0


In [41]:
education_salary = df[['EdLevel', 'ConvertedCompYearly']]
education_salary

,EdLevel,ConvertedCompYearly
701,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",273000.0
2731,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",95000.0
2846,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",20000.0
3242,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",NaN
4550,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",5300.0
6495,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",63986.0
6601,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",145000.0
11434,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",63808.0
11782,"Secondary school (e.g. American high school, G...",NaN
13996,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",165000.0


In [42]:
education_salary = education_salary.dropna(subset=['ConvertedCompYearly'])

In [43]:
education_salary_sorted = education_salary.sort_values(by='ConvertedCompYearly',ascending=False)
education_salary_sorted

,EdLevel,ConvertedCompYearly
24719,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",360000.0
701,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",273000.0
14301,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",230000.0
40192,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",220000.0
20399,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",190000.0
13996,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",165000.0
35580,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",154000.0
6601,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",145000.0
41253,"Secondary school (e.g. American high school, G...",127616.0
2731,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",95000.0


In [44]:
education_salary_sorted['EdLevel'].head(5)

24719    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)
701       Professional degree (JD, MD, Ph.D, Ed.D, etc.)
14301     Professional degree (JD, MD, Ph.D, Ed.D, etc.)
40192    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)
20399    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)
Name: EdLevel, dtype: object

In [45]:
top5_education = education_salary_sorted.head(5)
top5_education

,EdLevel,ConvertedCompYearly
24719,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",360000.0
701,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",273000.0
14301,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",230000.0
40192,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",220000.0
20399,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",190000.0


In [46]:
df[['Age', 'LanguageHaveWorkedWith']].head()

,Age,LanguageHaveWorkedWith
701,55-64 years old,C;C#;HTML/CSS;JavaScript;Python;SQL;TypeScript
2731,45-54 years old,Assembly;Bash/Shell (all shells);HTML/CSS;Java...
2846,65 years or older,MATLAB;Python
3242,55-64 years old,Python
4550,25-34 years old,Bash/Shell (all shells);Go;JavaScript;PowerShe...


In [47]:
age_total = df['Age'].value_counts()
age_total

Age
25-34 years old      11
35-44 years old       9
18-24 years old       8
65 years or older     7
45-54 years old       5
55-64 years old       4
Name: count, dtype: int64

In [48]:
python_df = df[df['LanguageHaveWorkedWith'].str.contains('Python', na=False)]
age_python = python_df['Age'].value_counts()

In [49]:
age_analysis = pd.concat([age_total, age_python], axis=1)
age_analysis.columns = ['TotalRespondents', 'PythonDevelopers']

In [50]:
age_analysis = age_analysis.fillna(0)

In [51]:
age_analysis['PythonPercent'] = (age_analysis['PythonDevelopers'] / age_analysis['TotalRespondents'] * 100).round(2)
age_analysis

,TotalRespondents,PythonDevelopers,PythonPercent
Age,,,
25-34 years old,11,8,72.73
35-44 years old,9,6,66.67
18-24 years old,8,5,62.50
65 years or older,7,6,85.71
45-54 years old,5,5,100.00
55-64 years old,4,3,75.00


In [52]:
age_analysis = age_analysis.sort_values('PythonPercent', ascending=False)
age_analysis

,TotalRespondents,PythonDevelopers,PythonPercent
Age,,,
45-54 years old,5,5,100.00
65 years or older,7,6,85.71
55-64 years old,4,3,75.00
25-34 years old,11,8,72.73
35-44 years old,9,6,66.67
18-24 years old,8,5,62.50


In [53]:
df[['ConvertedCompYearly', 'RemoteWork', 'Industry']].head()

,ConvertedCompYearly,RemoteWork,Industry
701,273000.0,Remote,Software Development
2731,95000.0,In-person,Other:
2846,20000.0,NaN,Software Development
3242,NaN,NaN,NaN
4550,5300.0,NaN,NaN


In [54]:
print(df['RemoteWork'].unique())

['Remote' 'In-person' nan
 'Hybrid (some in-person, leans heavy to flexibility)'
 'Hybrid (some remote, leans heavy to in-person)'
 'Your choice (very flexible, you can come in when you want or just as needed)']


In [55]:
# Розрахунок 75-го перцентиля річного доходу для визначення найбільш оплачуваних розробників
percentile_75 = df['ConvertedCompYearly'].quantile(0.75)

In [56]:
# Відбір високооплачуваних розробників (Top 25%) із віддаленим або гібридним форматом роботи
remote_options = [
    'Fully remote', 
    'Hybrid (some remote, leans heavy to in-person)', 
    'Hybrid (some in-person, leans heavy to remote)',
    'Your choice (very flexible, you can come in when you want or just as needed)']
high_paid_remote = df[(df['ConvertedCompYearly'] > percentile_75) & (df['RemoteWork'].isin(remote_options))]

In [57]:
# Підрахунок кількості високооплачуваних віддалених розробників за індустріями (Top-10)
top_industries = high_paid_remote['Industry'].value_counts()
print(top_industries.head(10))

Industry
Higher Education        1
Software Development    1
Healthcare              1
Name: count, dtype: int64
